In [ ]:
from valdpy import ValdAuth, ForeDecksAPI
from valdpy.utils import read_credentials

%load_ext autoreload
%autoreload 2

# ForceDecks API Example

This example demonstrates how to use the VALDPY package to access ForceDecks (force plate) test data.

## Prerequisites

1. Have VALDPY installed: `pip install -e ..`
2. Create a credentials file: Copy `vald_api_cred_TEMPLATE.txt` to `vald_api_cred.txt` and fill in your credentials
3. Ensure your VALD tenant has access to ForceDecks data

In [ ]:
# Read credentials from file
creds = read_credentials('vald_api_cred.txt')

client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']

print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

## Step 1: Authentication

Initialize the ValdAuth class to handle OAuth 2.0 authentication and manage tenant/profile information.

In [ ]:
# Initialize authentication object
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

Call `get_token()` to obtain an access token. This should be your first API call.

In [ ]:
# Get access token
token = auth.get_token()
print(f"Token obtained successfully: {token[:20]}...")

### Optional: Get Tenant Information

Retrieve information about your tenant, including available categories and groups.

In [ ]:
# Get all tenants accessible with your credentials
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")
for tenant in all_tenants[:3]:
    print(f"  - {tenant['name']} ({tenant['id']})")

In [ ]:
# Get tenant info
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

## Step 2: Get Categories and Groups

Retrieve available categories and groups to organize your profiles.

In [ ]:
# Get available categories
categories_df = auth.get_tenant_categories()
print("Available Categories:")
print(categories_df[['name', 'id']].to_string(index=False))

In [ ]:
# Get groups for the tenant
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].to_string(index=False))

## Admin Operations Reference

This notebook demonstrates all available tenant administration operations through the VALDPY library:

| Operation | Purpose | Caution |
|-----------|---------|---------|
| Create Category | Organize groups into logical containers | Non-destructive |
| Create Group | Create profile containers within categories | Non-destructive |
| Create Profile | Add new athlete records to the system | Non-destructive |
| Add Groups to Profile | Assign profile to one or more groups | Additive only |
| Overwrite Groups to Profile | Replace all group memberships for a profile | ⚠️ Removes existing memberships |
| Overwrite Profiles to Group | Replace all members of a group | ⚠️ Removes existing members |
| Remove Profiles from Group | Remove specific profiles from a group | Selective removal |
| Remove Groups from Profile | Remove profile from specific groups | Selective removal |
| Delete Group | Remove a group from the system | ⚠️ Destructive - cannot undo |

**Key Principles:**
- **Profiles** represent individual athletes
- **Groups** organize multiple profiles within a category
- **Categories** organize groups by type/purpose
- Remove/Delete operations are more restrictive than Add operations
- Use "Overwrite" operations carefully as they replace ALL current associations

In [ ]:
auth.create_category('temp4')

## Step 3: Manage Categories

**Create a new category** to organize your groups and profiles within your tenant.

**Usage:**
- `category_name` (str): Name of the new category to create
- Returns the new category object

**Example use case:** Create a category like "Teams", "Research Groups", or "Departments"

In [ ]:
auth.create_group('temp1','Uncategorised')

## Step 4: Manage Groups

**Create a new group** within a specific category. Groups are used to organize profiles.

**Usage:**
- `group_name` (str): Name of the new group
- `category_name` (str): Name of the category to place this group in
- Returns the new group object

**Example use case:** Create groups like "Athletes", "Control Group", or "Winter Sports Team"

In [ ]:
auth.get_group_profiles('Research',category_name='Team').head(20)

## Step 5: Retrieve Profiles from Groups

**Get all profiles within a specific group** organized by category.

**Usage:**
- `group_name` (str): Name of the group to query
- `category_name` (str): Name of the category containing the group
- Returns a DataFrame with all profiles in that group

**Example use case:** Retrieve all athlete profiles from a "Basketball Team" group to run batch analysis

In [ ]:
auth.get_profiles(['d40fe7d3-cdab-4107-87d9-0fafd0374f21'])

## Step 6: Query Profile Information

**Retrieve profile details** using multiple methods - by profile ID, ID list, or group name.

### Method 1: Get Profile by ID
Get a single profile using its unique ID. Returns a DataFrame with profile information.

In [ ]:
auth.get_profiles(profileIds=['d40fe7d3-cdab-4107-87d9-0fafd0374f21','2577f1d4-bd1d-4a22-8b1c-e64cf454fff0'])

### Method 2: Get Multiple Profiles by IDs
Retrieve multiple profiles by passing a list of profile IDs. Useful for bulk retrieval of specific athletes.

In [ ]:
auth.get_profiles(groupName='Research')

### Method 3: Get All Profiles in a Group
Retrieve all profiles that belong to a specific group by group name.

In [ ]:
auth.create_profile(givenName='Research', familyName='15', dateOfBirth='01/01/2000', sex='NotApplicable')

## Step 7: Create Profiles

**Create a new athlete profile** in your VALD tenant. Store basic information about the profile.

**Parameters:**
- `givenName` (str): First name of the athlete
- `familyName` (str): Last name of the athlete
- `dateOfBirth` (str): Date of birth in DD/MM/YYYY format
- `sex` (str): Sex/Gender ("Male", "Female", "NotApplicable")
- Returns the new profile object with a unique profile ID

**Example use case:** Onboard a new athlete into the system for performance testing

In [ ]:
auth.add_groups_to_profile(profileId='5873f4f9-5641-43b2-9d76-4b96dd0edc5e',groupNames=['Research'])

## Step 8: Manage Profile Memberships

**Add groups to a profile** - assign a profile to one or more groups. This is typically done after creating a new profile.

**Usage:**
- `profileId` (str): ID of the profile to modify
- `groupNames` (list): List of group names to add the profile to
- `categoryName` (optional, list): Categories corresponding to each group
- If a profile is already in a group, it won't be added again

**Example use case:** Assign a newly created athlete to "Basketball Team" and "Conditioning Group"

### Add Multiple Groups with Categories
Add a profile to multiple groups across different categories in a single operation.

In [ ]:
auth.add_groups_to_profile(profileId='5873f4f9-5641-43b2-9d76-4b96dd0edc5e',groupNames=['Research','temp1'],categoryName=['Team','Uncategorised'])

### Overwrite Profile's Group Membership (⚠️ Caution)
**Replace all group memberships** for a profile with a new set of groups. This removes the profile from any groups not in the new list.

**Warning:** This operation removes the profile from existing groups. Use carefully!

**Usage:**
- `profileId` (str): ID of the profile to modify
- `groupNames` (list): New list of groups (replaces current membership)
- `categoryNames` (list): Categories corresponding to each group

**Example use case:** Reorganize a profile's group assignments when an athlete switches teams

In [ ]:
auth.overwrite_groups_to_profile(profileId='d7f89f7a-831b-480d-8bef-efc8884d0c5c',groupNames=['Research','temp1'],categoryNames=['Team','Uncategorised'])

In [ ]:
auth.profile_df['profileId'].values.tolist()

## Step 9: Manage Group Memberships

**Access the internal profile DataFrame** to work with profile IDs. This DataFrame is stored in `auth.profile_df` and contains all loaded profiles with their metadata.

### Overwrite Group Membership (⚠️ Caution)
**Replace all profiles in a group** with a new set of profiles. Existing profiles in the group will be removed.

**Warning:** This operation removes existing profiles from the group. Use carefully!

**Usage:**
- `groupName` (str): Name of the group to modify
- `profileIds` (list): New list of profile IDs (replaces current members)
- `categoryName` (str): Category containing the group

**Example use case:** Update group roster when multiple athletes change teams

In [ ]:
auth.overwrite_profiles_to_group(groupName='temp1', profileIds=auth.profile_df['profileId'].values.tolist(), categoryName='Uncategorised')

### Remove Profiles from Group
**Remove specific profiles from a group** while keeping other members. Profiles remain in the system but are no longer associated with this group.

**Usage:**
- `groupName` (str): Name of the group
- `profileIds` (list): List of profile IDs to remove from the group
- `categoryName` (str): Category containing the group

**Example use case:** Remove graduated athletes from an active roster

In [ ]:
auth.remove_profiles_from_group(groupName='temp1', profileIds=auth.profile_df['profileId'].values.tolist()[1:], categoryName='Uncategorised')

### Remove Groups from Profile
**Remove a profile from specific groups** while keeping it in other groups. The profile remains in the system.

**Usage:**
- `profileId` (str): ID of the profile
- `groupNames` (list): List of group names to remove the profile from
- `categoryNames` (list): Categories corresponding to each group

**Example use case:** Remove an athlete from a conditioning group while keeping them in their main sports team

In [ ]:
auth.remove_groups_from_profile(profileId='d40fe7d3-cdab-4107-87d9-0fafd0374f21', groupNames=['temp1'], categoryNames=['Uncategorised'])

## Step 10: Clean Up - Delete Groups (⚠️ Destructive)

**Delete a group** from your tenant. This removes the group and all its associations, but profiles are NOT deleted.

**Warning:** This is a destructive operation that cannot be undone. Ensure you have backups before proceeding.

**Usage:**
- `group_name` (str): Name of the group to delete
- `category_name` (str): Category containing the group

**Example use case:** Remove temporary testing groups after completing an assessment

In [ ]:
auth.delete_group('temp1','Uncategorised')